# Observabilidade de dados — política de acordos

Este notebook une os resultados dos processos aos subsídios disponibilizados pelo banco. O objetivo é encontrar padrões para investigação, com gráficos Plotly interativos. **Correlação não prova causalidade**: use os recortes para priorizar hipóteses e revisão jurídica.

Fluxo: `CSVs brutos → padronização e validação → dataset por processo → métricas e gráficos → revisão da política`.

## Pré-requisitos

No ambiente do projeto, instale as dependências uma vez: `py -m pip install -r requirements.txt`. Depois execute **Run All**. O notebook localiza os CSVs dentro de `data/`, mesmo quando aberto a partir de `notebooks/`.

In [1]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 50)

ROOT = Path.cwd().resolve()
while not (ROOT / 'data').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
assert DATA_DIR.exists(), 'Abra o notebook dentro do repositório ShiftHappens.'

SUBSIDIES_FILE = next(DATA_DIR.glob('*Candidatos.csv'))
RESULTS_FILE = next(DATA_DIR.glob('*Resultados*.csv'))
print(f'Repositório: {ROOT}')
print(f'Subsídios: {SUBSIDIES_FILE.name}')
print(f'Resultados: {RESULTS_FILE.name}')

Repositório: C:\Users\pedro\OneDrive\Área de Trabalho\ENTER\ShiftHappens
Subsídios: Hackaton_Enter_Base_Candidatos.csv
Resultados: Hackaton_Enter_Base_Candidatos.xlsx - Resultados dos processos.csv


In [2]:
def normalizar_nome(valor):
    valor = unicodedata.normalize('NFKD', str(valor)).encode('ascii', 'ignore').decode().lower().strip()
    return re.sub(r'[^a-z0-9]+', '_', valor).strip('_')

def moeda_brl(serie):
    return pd.to_numeric(
        serie.astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce'
    )

# O primeiro registro do arquivo de subsídios descreve o significado de 0/1.
subsidios = pd.read_csv(SUBSIDIES_FILE, skiprows=1, encoding='utf-8-sig')
resultados = pd.read_csv(RESULTS_FILE, encoding='utf-8-sig')

subsidios.columns = [normalizar_nome(c) for c in subsidios.columns]
resultados.columns = [normalizar_nome(c) for c in resultados.columns]

# Corrige a variação de nomenclatura entre os dois arquivos.
subsidios = subsidios.rename(columns={'numero_do_processos': 'numero_processo'})
resultados = resultados.rename(columns={'numero_do_processo': 'numero_processo'})

DOCS = [
    'contrato', 'extrato', 'comprovante_de_credito', 'dossie',
    'demonstrativo_de_evolucao_da_divida', 'laudo_referenciado'
]
required_results = ['numero_processo', 'uf', 'assunto', 'sub_assunto', 'resultado_macro',
                    'resultado_micro', 'valor_da_causa', 'valor_da_condenacao_indenizacao']
missing = set(DOCS).difference(subsidios.columns) | set(required_results).difference(resultados.columns)
assert not missing, f'Colunas obrigatórias ausentes: {sorted(missing)}'

for col in DOCS:
    subsidios[col] = pd.to_numeric(subsidios[col], errors='coerce').astype('Int64')
resultados['valor_da_causa'] = moeda_brl(resultados['valor_da_causa'])
resultados['valor_da_condenacao_indenizacao'] = moeda_brl(resultados['valor_da_condenacao_indenizacao'])

# Métricas de qualidade antes do join.
quality = pd.DataFrame({
    'fonte': ['resultados', 'subsídios'],
    'linhas': [len(resultados), len(subsidios)],
    'ids_duplicados': [resultados['numero_processo'].duplicated().sum(), subsidios['numero_processo'].duplicated().sum()],
    'id_nulo': [resultados['numero_processo'].isna().sum(), subsidios['numero_processo'].isna().sum()]
})
quality

,fonte,linhas,ids_duplicados,id_nulo
0,resultados,60000,0,0
1,subsídios,60000,0,0


In [3]:
# O indicador preserva a rastreabilidade da cobertura do join.
df = resultados.merge(subsidios[['numero_processo', *DOCS]], on='numero_processo', how='left', indicator=True)
df['teve_nao_exito'] = df['resultado_macro'].astype(str).str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('utf-8').str.lower().eq('nao exito')
df['qtd_subsidios'] = df[DOCS].sum(axis=1)
df['razao_condenacao_causa'] = np.where(df['valor_da_causa'] > 0, df['valor_da_condenacao_indenizacao'] / df['valor_da_causa'], np.nan)

join_rate = (df['_merge'] == 'both').mean()
fig = go.Figure(go.Indicator(
    mode='number+delta', value=join_rate * 100, number={'suffix': '%', 'valueformat': '.1f'},
    title={'text': 'Cobertura do join: resultados com subsídios'},
    delta={'reference': 100, 'suffix': ' p.p.', 'valueformat': '.1f'}
))
fig.update_layout(height=250, template='plotly_white')
fig.show()

df['_merge'].value_counts().rename_axis('situação').to_frame('processos')

,processos
situação,
both,60000
left_only,0
right_only,0


In [4]:
# Completude: frequência individual de cada documento de defesa.
coverage = (df[DOCS].mean().mul(100).sort_values().rename('cobertura_pct').reset_index()
            .rename(columns={'index': 'subsídio'}))
fig = px.bar(coverage, x='cobertura_pct', y='subsídio', orientation='h', text_auto='.1f',
             title='Cobertura de subsídios por processo', labels={'cobertura_pct': 'Cobertura (%)'})
fig.update_layout(template='plotly_white', yaxis={'categoryorder': 'total ascending'})
fig.update_xaxes(range=[0, 100])
fig.show()

px.histogram(df, x='qtd_subsidios', nbins=7, title='Quantidade de subsídios por processo',
             labels={'qtd_subsidios': 'Número de subsídios'}).update_layout(template='plotly_white').show()

curva = (df.groupby('qtd_subsidios', as_index=False)
           .agg(processos=('numero_processo', 'size'), taxa_exito=('teve_nao_exito', lambda s: 1 - s.mean()),
                condenacao_media=('valor_da_condenacao_indenizacao', 'mean')))
curva['taxa_exito_pct'] = curva['taxa_exito'] * 100
fig = px.line(curva, x='qtd_subsidios', y='taxa_exito_pct', markers=True,
              hover_data=['processos', 'condenacao_media'], title='Curva de êxito por completude documental',
              labels={'qtd_subsidios': 'Número de subsídios', 'taxa_exito_pct': 'Taxa de êxito (%)'})
fig.update_layout(template='plotly_white')
fig.update_yaxes(range=[0, 100])
fig.show()

In [5]:
# Para variáveis binárias, Pearson equivale ao coeficiente phi.
corr = df[DOCS].astype(float).corr()
fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Correlação entre presença de subsídios (phi)')
fig.update_layout(template='plotly_white', height=650)
fig.show()

pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
         .sort_values(key=lambda s: s.abs(), ascending=False).rename('correlação').reset_index())
pairs.columns = ['subsídio_a', 'subsídio_b', 'correlação']
pairs.head(10)

,subsídio_a,subsídio_b,correlação
0,contrato,extrato,0.318319
1,contrato,comprovante_de_credito,0.174386
2,extrato,comprovante_de_credito,0.152407
3,contrato,demonstrativo_de_evolucao_da_divida,0.067534
4,extrato,demonstrativo_de_evolucao_da_divida,0.056274
5,comprovante_de_credito,demonstrativo_de_evolucao_da_divida,0.029454
6,extrato,laudo_referenciado,-0.008285
7,comprovante_de_credito,dossie,0.006454
8,demonstrativo_de_evolucao_da_divida,laudo_referenciado,-0.005303
9,contrato,dossie,0.003060


In [6]:
# Associação observacional: compare taxas de não êxito com e sem cada subsídio.
risk_by_doc = (df.melt(id_vars='teve_nao_exito', value_vars=DOCS, var_name='subsídio', value_name='disponível')
                 .dropna().groupby(['subsídio', 'disponível'], as_index=False)['teve_nao_exito'].mean())
risk_by_doc['taxa_nao_exito_pct'] = risk_by_doc['teve_nao_exito'] * 100
risk_by_doc['disponível'] = risk_by_doc['disponível'].map({0: 'Não', 1: 'Sim'})
fig = px.bar(risk_by_doc, x='subsídio', y='taxa_nao_exito_pct', color='disponível', barmode='group',
             title='Taxa de não êxito por disponibilidade de subsídio',
             labels={'taxa_nao_exito_pct': 'Taxa de não êxito (%)'})
fig.update_layout(template='plotly_white', xaxis_tickangle=-30)
fig.show()

In [7]:
# Recorte gerencial: volume, taxa de não êxito e exposição financeira por UF/subassunto.
segmento = (df.groupby(['uf', 'sub_assunto'], dropna=False)
             .agg(processos=('numero_processo', 'size'), taxa_nao_exito=('teve_nao_exito', 'mean'),
                  condenacao_media=('valor_da_condenacao_indenizacao', 'mean'),
                  condenacao_total=('valor_da_condenacao_indenizacao', 'sum'))
             .reset_index())
segmento['taxa_nao_exito_pct'] = segmento['taxa_nao_exito'] * 100
fig = px.scatter(segmento, x='taxa_nao_exito_pct', y='condenacao_media', size='processos', color='sub_assunto',
                 hover_name='uf', hover_data=['processos', 'condenacao_total'],
                 title='Risco por UF e subassunto: taxa de não êxito × condenação média',
                 labels={'taxa_nao_exito_pct': 'Taxa de não êxito (%)', 'condenacao_media': 'Condenação média (R$)'})
fig.update_layout(template='plotly_white')
fig.show()

segmento.sort_values('condenacao_total', ascending=False).head(15)

,uf,sub_assunto,processos,taxa_nao_exito,condenacao_media,condenacao_total,taxa_nao_exito_pct
5,AM,Golpe,1625,0.540308,6866.375735,11157860.57,54.030769
7,AP,Golpe,1616,0.556312,6900.576658,11151331.88,55.631188
9,BA,Golpe,1571,0.413113,4989.694755,7838810.46,41.311267
43,RS,Golpe,1616,0.439356,4838.886256,7819640.19,43.935644
17,GO,Golpe,1601,0.435978,4827.491062,7728813.19,43.597751
37,RJ,Golpe,1616,0.396658,4080.849301,6594652.47,39.665842
31,PE,Golpe,1623,0.362908,3947.293543,6406457.42,36.290819
13,DF,Golpe,1600,0.381250,3997.777100,6396443.36,38.125000
15,ES,Golpe,1547,0.393665,4134.019140,6395327.61,39.366516
3,AL,Golpe,1596,0.372807,4006.136209,6393793.39,37.280702


In [8]:
# Cramér's V mede associação entre variáveis categóricas (0 a 1, não-direcional).
def cramers_v(a, b):
    table = pd.crosstab(a, b)
    n = table.values.sum()
    if n == 0 or min(table.shape) < 2:
        return np.nan
    expected = np.outer(table.sum(1), table.sum(0)) / n
    chi2 = ((table - expected) ** 2 / expected).to_numpy().sum()
    return np.sqrt((chi2 / n) / min(table.shape[0] - 1, table.shape[1] - 1))

categorias = [c for c in ['uf', 'assunto', 'sub_assunto', 'resultado_macro', 'resultado_micro']
              if df[c].nunique(dropna=True) > 1]
v = pd.DataFrame([[cramers_v(df[a], df[b]) for b in categorias] for a in categorias],
                 index=categorias, columns=categorias)
fig = px.imshow(v, text_auto='.2f', color_continuous_scale='Blues', zmin=0, zmax=1,
                title="Associação entre categorias (Cramér's V)")
fig.update_layout(template='plotly_white', height=600)
fig.show()

C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be key

C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be key

C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be key

C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n
C:\Users\pedro\AppData\Local\Temp\ipykernel_14800\4165576780.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  expected = np.outer(table.sum(1), table.sum(0)) / n


In [9]:
# Plano de pipeline: uma visão executável do caminho mínimo até o monitoramento.
nodes = ['Resultados CSV', 'Subsídios CSV', 'Validação + padronização', 'Dataset por processo',
         'Features de evidência', 'KPIs de desfecho', 'Notebook / dashboard', 'Alertas e revisão']
links = [(0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 6), (5, 6), (6, 7)]
fig = go.Figure(go.Sankey(
    node=dict(label=nodes, pad=20, thickness=20),
    link=dict(source=[s for s, t in links], target=[t for s, t in links], value=[1] * len(links))
))
fig.update_layout(title='Pipeline mínimo de observabilidade', template='plotly_white', height=420)
fig.show()

In [10]:
# Insights automáticos para a primeira conversa com o time jurídico e de dados.
top_pair = pairs.iloc[0]
top_segment = segmento.sort_values('condenacao_total', ascending=False).iloc[0]
doc_delta = (risk_by_doc.pivot(index='subsídio', columns='disponível', values='taxa_nao_exito_pct')
             .assign(delta_pp=lambda x: x.get('Sim', np.nan) - x.get('Não', np.nan)))
most_changed_doc = doc_delta['delta_pp'].abs().idxmax()

print(f'• {len(df):,} processos analisados; cobertura do join: {join_rate:.1%}.')
print(f"• Maior correlação de presença: {top_pair['subsídio_a']} × {top_pair['subsídio_b']} ({top_pair['correlação']:.2f}).")
print(f'• Maior diferença observada de taxa de não êxito: {most_changed_doc} ({doc_delta.loc[most_changed_doc, "delta_pp"]:+.1f} p.p. com vs. sem).')
print(f"• Maior exposição total: UF {top_segment['uf']} / {top_segment['sub_assunto']} — R$ {top_segment['condenacao_total']:,.0f}.")
print('Próximo passo: validar estes recortes com amostra jurídica e incluir eventos de recomendação, decisão, oferta e aceite.')

• 60,000 processos analisados; cobertura do join: 100.0%.
• Maior correlação de presença: contrato × extrato (0.32).
• Maior diferença observada de taxa de não êxito: extrato (-62.9 p.p. com vs. sem).
• Maior exposição total: UF AM / Golpe — R$ 11,157,861.
Próximo passo: validar estes recortes com amostra jurídica e incluir eventos de recomendação, decisão, oferta e aceite.
